# Gate 4 - Indian-English Accent Conversion Strength Sweep

**Purpose:** Verify that AccentConverter produces monotonically increasing accent change at multiple strength levels.

**Pass criteria:**
- Identity shift at strength=0 ≈ 0
- Identity shift at strength=1 > 0.15
- Identity shift increases monotonically
- mel L1 < 0.5 at all strengths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
!apt-get update -qq
!apt-get install -y -qq espeak-ng > /dev/null 2>&1
print('espeak-ng installed')

In [ ]:
import os, subprocess, sys, json, time, warnings
from pathlib import Path

warnings.simplefilter('ignore')

ACCENTEDGE_DIR = '/content/accentedge'
FA_CODEC_DIR = '/content/FAcodec'
GATE_DIR = '/content/gate4_artifacts'
DRIVE_BASE = '/content/drive/MyDrive/accentedge/runs'

# Clone repo if not present
if not os.path.exists(ACCENTEDGE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ayushmh/accentedge.git', ACCENTEDGE_DIR],
                   check=True)

os.chdir(ACCENTEDGE_DIR)
sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")

# Clone FAcodec if not present
if not os.path.exists(FA_CODEC_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Plachtaa/FAcodec.git', FA_CODEC_DIR],
                   check=True)

# Check FAcodec checkpoint
FA_CODEC_CKPT = os.environ.get('FA_CODEC_CKPT', '')
if not FA_CODEC_CKPT or not os.path.exists(FA_CODEC_CKPT):
    for candidate in [
        '/content/checkpoints/facodec.pth',
        f'{ACCENTEDGE_DIR}/checkpoints/facodec.pth',
    ]:
        if os.path.exists(candidate):
            FA_CODEC_CKPT = candidate
            break

print(f'AccentEdge: {ACCENTEDGE_DIR}')
print(f'FAcodec: {FA_CODEC_DIR}')
print(f'FAcodec checkpoint: {FA_CODEC_CKPT or "NOT FOUND"}')

In [ ]:
!pip install -q torch torchaudio transformers faster-whisper phonemizer speechbrain librosa jiwer pyyaml soundfile scipy numpy matplotlib einops huggingface-hub munch pyworld

# Verify imports
import torch, torchaudio, numpy as np
print(f'torch: {torch.__version__}')
print(f'torchaudio: {torchaudio.__version__}')
print(f'numpy: {np.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Download test audio
import os, urllib.request

TEST_DIR = '/content/test_audio'
os.makedirs(TEST_DIR, exist_ok=True)

# L2-ARCTIC Hindi samples (download directly)
L2ARCTIC_BASE = 'https://huggingface.co/datasets/tonypan101/L2-Arctic-Test/resolve/main'
# For demo, generate synthetic test tones
import torchaudio

for i, freq in enumerate([220, 440, 880]):
    t = torch.linspace(0, 2.0, 48000)
    wav = torch.sin(2 * 3.14159 * freq * t).unsqueeze(0) * 0.3
    torchaudio.save(f'{TEST_DIR}/tone_{freq}hz.wav', wav, 24000)

print(f'Test audio: {TEST_DIR}')
!ls -la {TEST_DIR}

In [ ]:
# Run Gate 4 strength sweep
import subprocess

N_SAMPLES = 5
GATE_DIR = '/content/gate4_artifacts'
DRIVE_OUT = f'{DRIVE_BASE}/gate4'

# Setup environment
env = os.environ.copy()
env['PYTHONPATH'] = f'/content/accentedge/src:{env.get('PYTHONPATH', '')}'

cmd = [
    sys.executable, f'{ACCENTEDGE_DIR}/scripts/gate4_strength_sweep.py',
    '--device', 'cuda',
    '--n-samples', str(N_SAMPLES),
    '--output-dir', GATE_DIR,
    '--facodec-ckpt', FA_CODEC_CKPT or '',
    '--no-wer',  # Skip WER for faster execution
]

print(f'Running: {' '.join(cmd)}')
result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:2000])

if result.returncode != 0:
    print(f'Gate 4 exited with code {result.returncode}')

## Results Summary

Gate 4 strength sweep complete. Below are the strength curves and pass/fail analysis.

In [ ]:
import json
from pathlib import Path

GATE_DIR = '/content/gate4_artifacts'

# Load results
with open(f'{GATE_DIR}/strength_curves.json') as f:
    curves_data = json.load(f)

curves = curves_data['strength_curves']
gate = curves_data['gate_evaluation']
strengths = [float(s) for s in curves.keys()]

print('Strength curves loaded:')
for s in strengths:
    c = curves[str(s)]
    print(f'  {s}: mel_l1={c['mel_l1_mean']:.4f}, id_shift={c['identity_shift_mean']:.4f}, n={c['n_samples']}')

print(f'\nGate passed: {gate['passed']}')
if gate['failures']:
    for f in gate['failures']:
        print(f'  FAIL: {f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

strengths_list = [float(s) for s in curves.keys()]
id_shifts = [curves[str(s)]['identity_shift_mean'] for s in strengths_list]
id_stds = [curves[str(s)]['identity_shift_std'] for s in strengths_list]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strengths_list, id_shifts, 'b-o', linewidth=2, markersize=8, label='Mean identity shift')
ax.fill_between(strengths_list,
                 [m - s for m, s in zip(id_shifts, id_stds)],
                 [m + s for m, s in zip(id_shifts, id_stds)],
                 alpha=0.2, color='blue')
ax.axhline(y=0.15, color='r', linestyle='--', label='Threshold (0.15)')
ax.axhline(y=0.0, color='g', linestyle='--', alpha=0.5)
ax.set_xlabel('Strength', fontsize=12)
ax.set_ylabel('Identity Shift (cosine distance)', fontsize=12)
ax.set_title('Identity Shift vs Strength', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

mel_means = [curves[str(s)]['mel_l1_mean'] for s in strengths_list]
mel_stds = [curves[str(s)]['mel_l1_std'] for s in strengths_list]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strengths_list, mel_means, 'g-s', linewidth=2, markersize=8, label='Mean mel L1')
ax.fill_between(strengths_list,
                 [m - s for m, s in zip(mel_means, mel_stds)],
                 [m + s for m, s in zip(mel_means, mel_stds)],
                 alpha=0.2, color='green')
ax.axhline(y=0.5, color='r', linestyle='--', label='Threshold (0.5)')
ax.set_xlabel('Strength', fontsize=12)
ax.set_ylabel('Mel L1 Distance', fontsize=12)
ax.set_title('Acoustic Quality (Mel L1) vs Strength', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

if curves['0.0'].get('wer_mean') is not None:
    wer_means = [curves[str(s)]['wer_mean'] for s in strengths_list]
    wer_stds = [curves[str(s)]['wer_std'] for s in strengths_list]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(strengths_list, wer_means, 'm-^', linewidth=2, markersize=8, label='Mean WER')
    ax.fill_between(strengths_list,
                     [m - s for m, s in zip(wer_means, wer_stds)],
                     [m + s for m, s in zip(wer_means, wer_stds)],
                     alpha=0.2, color='magenta')
    ax.set_xlabel('Strength', fontsize=12)
    ax.set_ylabel('Word Error Rate', fontsize=12)
    ax.set_title('Content Preservation (WER) vs Strength', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('WER data not available in strength_curves.json')

In [ ]:
from IPython.display import Audio, display
import torchaudio

audio_dir = Path(GATE_DIR) / 'audio'
if audio_dir.exists():
    wav_files = sorted(list(audio_dir.glob('*.wav')))
    print(f'Found {len(wav_files)} audio files')
    for wav_path in wav_files[:6]:
        print(f'\n{wav_path.name}:')
        display(Audio(filename=str(wav_path), autoplay=False))
else:
    print('Audio directory not found')

In [ ]:
import pandas as pd

rows = []
for s in strengths_list:
    c = curves[str(s)]
    rows.append({
        'Strength': s,
        'mel_L1_mean': c['mel_l1_mean'],
        'mel_L1_std': c['mel_l1_std'],
        'id_shift_mean': c['identity_shift_mean'],
        'id_shift_std': c['identity_shift_std'],
        'duration_ratio_mean': c['duration_ratio_mean'],
        'snr_db_mean': c.get('snr_db_mean', None),
        'wer_mean': c.get('wer_mean', None),
        'n_samples': c['n_samples'],
    })

df = pd.DataFrame(rows)
display(df)

## Summary and Next Steps

Gate 4 strength sweep verifies that accent conversion strength controls the degree of accent change. Review the identity shift curve to confirm monotonic behavior. If Gate 4 passes, proceed to Phase 2 training with strength-conditioned diffusion.